# Create Datasets
This is the code that I used for creating different datasets. How to get valid patients csv (T1 surface and T1 structural) is explained in artherosclerosis branch.

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR,MultiStepLR
from PIL import Image
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import pandas as pd
import yaml
import time
import numpy as np
import pandas as pd
from data.dataset_ori1 import PatientDataset

/nethome/kcni/evhuang/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.2' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## T1 surface
'valid_patients' are a list of patients who had MRI scans. I filtered their icd code to get the diagnosis of cerebrovascular diseases.

In [39]:
valid_patients = '{your path to the list of valid participants with all diagnoses}' 
valid_patients = pd.read_csv(valid_patients)

cvb = valid_patients[['eid','I67']]
cvb.to_csv('CVB_Diagnoses.csv')

In [40]:
# Filter the data into positive and negative patients
positive_cases = cvb[cvb['I67'] == 1]
negative_cases = cvb[cvb['I67'] == 0]

I defined a function here to create datasets with different numbers of patients in train, validation and test datasets. (Feel free to change the ratio)  
The entire dataset is split in train_data, val_data, test_data with the ratio of **7:2:1**  
train_data: 25509  
val_data: 7288  
test_data: 3644  
note: These datasets are **imbalanced**

In [19]:
def stratified_sample(data, train_size, val_size, test_size, random_state=42):
    train_pos_sample = positive_cases.sample(frac=train_size, random_state=random_state)
    train_neg_sample = negative_cases.sample(frac=train_size, random_state=random_state)
    
    remaining_positive_cases = positive_cases.drop(train_pos_sample.index)
    remaining_negative_cases = negative_cases.drop(train_neg_sample.index)
    
    val_pos_sample = remaining_positive_cases.sample(frac=val_size / (val_size + test_size), random_state=random_state)
    val_neg_sample = remaining_negative_cases.sample(frac=val_size / (val_size + test_size), random_state=random_state)
    
    test_pos_sample = remaining_positive_cases.drop(val_pos_sample.index)
    test_neg_sample = remaining_negative_cases.drop(val_neg_sample.index)
    
    train_data = pd.concat([train_pos_sample, train_neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    val_data = pd.concat([val_pos_sample, val_neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_data = pd.concat([test_pos_sample, test_neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return train_data, val_data, test_data

# Sample sizes (can be adjusted as needed)
train_size = 0.7
val_size = 0.2
test_size = 0.1

train_data, val_data, test_data = stratified_sample(cvb, train_size, val_size, test_size)

train_data.to_csv('train_data.csv', index=False)
val_data.to_csv('val_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)

I sampled some patients to create balanced datasets  
train:val:test = **7:2:1**  
train_1: 280  
val_1: 80  
test_1: 40
Note: These datasets are **balanced**

In [20]:
# Sample cases for training
train_pos_sample = positive_cases.sample(n=140, random_state=42)
train_neg_sample = negative_cases.sample(n=140, random_state=42)

# Ensure remaining data is used for val/test splits
remaining_positive_cases = positive_cases.drop(train_pos_sample.index)
remaining_negative_cases = negative_cases.drop(train_neg_sample.index)

# Sample cases for validation
val_pos_sample = remaining_positive_cases.sample(n=40, random_state=42)
val_neg_sample = remaining_negative_cases.sample(n=40, random_state=42)

# Sample cases for testing
test_pos_sample = remaining_positive_cases.drop(val_pos_sample.index).sample(n=20, random_state=42)
test_neg_sample = remaining_negative_cases.drop(val_neg_sample.index).sample(n=20, random_state=42)

# Combine and shuffle training samples
train_sample = pd.concat([train_pos_sample, train_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle validation samples
val_sample = pd.concat([val_pos_sample, val_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle test samples
test_sample = pd.concat([test_pos_sample, test_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

train_sample = train_sample[['eid', 'I67']]
val_sample = val_sample[['eid', 'I67']]
test_sample = test_sample[['eid', 'I67']]

# Save the datasets
train_sample.to_csv('train_1.csv', index=False)
val_sample.to_csv('val_1.csv', index=False)
test_sample.to_csv('test_1.csv', index=False)

I created imbalanced dataset with **case:control = 1:1**  
train:val:test = **7:2:1**   
train_ib: 1400  
val_ib: 400  
test_ib: 200

In [21]:
# create imbalanced dataset
train_pos_sample = positive_cases.sample(n=140, random_state=42)
train_neg_sample = negative_cases.sample(n=1260, random_state=42)

# Ensure remaining data is used for val/test splits
remaining_positive_cases = positive_cases.drop(train_pos_sample.index)
remaining_negative_cases = negative_cases.drop(train_neg_sample.index)

# Sample cases for validation
val_pos_sample = remaining_positive_cases.sample(n=40, random_state=42)
val_neg_sample = remaining_negative_cases.sample(n=360, random_state=42)

# Sample cases for testing
test_pos_sample = remaining_positive_cases.drop(val_pos_sample.index).sample(n=20, random_state=42)
test_neg_sample = remaining_negative_cases.drop(val_neg_sample.index).sample(n=180, random_state=42)

# Combine and shuffle training samples
train_sample = pd.concat([train_pos_sample, train_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle validation samples
val_sample = pd.concat([val_pos_sample, val_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle test samples
test_sample = pd.concat([test_pos_sample, test_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

train_sample = train_sample[['eid', 'I67']]
val_sample = val_sample[['eid', 'I67']]
test_sample = test_sample[['eid', 'I67']]

# Save the datasets
train_sample.to_csv('train_ib.csv', index=False)
val_sample.to_csv('val_ib.csv', index=False)
test_sample.to_csv('test_ib.csv', index=False)

I sampled some patients to get smaller **balanced** datasets for training  
train:val:test = **7:2:1**  
train_sample: 140  
val_sample: 40  
test_sample: 20

In [27]:
# Sample cases for training
train_pos_sample = positive_cases.sample(n=70, random_state=42)
train_neg_sample = negative_cases.sample(n=70, random_state=42)

# Ensure remaining data is used for val/test splits
remaining_positive_cases = positive_cases.drop(train_pos_sample.index)
remaining_negative_cases = negative_cases.drop(train_neg_sample.index)

# Sample cases for validation
val_pos_sample = remaining_positive_cases.sample(n=20, random_state=42)
val_neg_sample = remaining_negative_cases.sample(n=20, random_state=42)

# Sample cases for testing
test_pos_sample = remaining_positive_cases.drop(val_pos_sample.index).sample(n=10, random_state=42)
test_neg_sample = remaining_negative_cases.drop(val_neg_sample.index).sample(n=10, random_state=42)

# Combine and shuffle training samples
train_sample = pd.concat([train_pos_sample, train_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle validation samples
val_sample = pd.concat([val_pos_sample, val_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle test samples
test_sample = pd.concat([test_pos_sample, test_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

train_sample = train_sample[['eid', 'I67']]
val_sample = val_sample[['eid', 'I67']]
test_sample = test_sample[['eid', 'I67']]

# Save the datasets
train_sample.to_csv('train_sample.csv', index=False)
val_sample.to_csv('val_sample.csv', index=False)
test_sample.to_csv('test_sample.csv', index=False)

## T1 structural
Since the list of valid patients is different from T1 surface, I needed a new list.

In [8]:
# create cropped dataset
valid_struct = '{your path to the list of valid participants with all diagnoses}' # put your path here 
valid_struct = pd.read_csv(valid_struct)

In [11]:
icd_struct = valid_struct[['eid', 'I67']]
icd_struct.to_csv('icd_struct.csv', index=False)

I sampled some patients with T1 stuctural to create datasets  
train:val:test = **5:1:1**  
train_crop: 500  
val_crop: 100  
test_crop: 100

In [ ]:
# Filter the data into positive and negative patients
positive_cases = valid_struct[valid_struct['I67'] == 1]
negative_cases = valid_struct[valid_struct['I67'] == 0]

# Sample cases for training
train_pos_sample = positive_cases.sample(n=50, random_state=42)
train_neg_sample = negative_cases.sample(n=450, random_state=42)

# Ensure remaining data is used for val/test splits
remaining_positive_cases = positive_cases.drop(train_pos_sample.index)
remaining_negative_cases = negative_cases.drop(train_neg_sample.index)

# Sample cases for validation
val_pos_sample = remaining_positive_cases.sample(n=10, random_state=42)
val_neg_sample = remaining_negative_cases.sample(n=90, random_state=42)

# Sample cases for testing
test_pos_sample = remaining_positive_cases.drop(val_pos_sample.index).sample(n=10, random_state=42)
test_neg_sample = remaining_negative_cases.drop(val_neg_sample.index).sample(n=90, random_state=42)

# Combine and shuffle training samples
train_sample = pd.concat([train_pos_sample, train_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle validation samples
val_sample = pd.concat([val_pos_sample, val_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Combine and shuffle test samples
test_sample = pd.concat([test_pos_sample, test_neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

train_sample = train_sample[['eid', 'I67']]
val_sample = val_sample[['eid', 'I67']]
test_sample = test_sample[['eid', 'I67']]

# Save the datasets
train_sample.to_csv('train_crop.csv', index=False)
val_sample.to_csv('val_crop.csv', index=False)
test_sample.to_csv('test_crop.csv', index=False)